# Main Fig 3 — Iso-compute Heatmap

**Data**: `heatmap_df_test.csv`  
**Tasks**: main 5  
**Head**: Transformer  
**Layout**: 5 rows × 1 column (full 7-in width per heatmap)  

Heatmap cell = AUROC (%) at that (context, K) combination.  Dashed lines = iso-compute budgets.

**Tip**: `ROW_H` controls row height. Increase for more vertical breathing room, decrease for a more compact figure.

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    """Walk up from CWD until we find a directory containing final_results/."""
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    # Explicit fallback (edit this if auto-detect fails)
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path (notebooks/utils/)
_nb_dir = PAPER_FIGURES / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL, FONT_ANNOT, FONT_BASE, FONT_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

# ── Confirm the workspace root is correct ─────────────────────────────────────
_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND — edit _find_workspace() fallback'}")

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
HEAD   = "transformer"
TASKS  = MAIN_TASKS   # 5 tasks → 2-2-1 layout
METRIC = "auroc"
ROW_H  = 2.0   # inches per row — increase for more vertical space

hmaps = {t: load_heatmap("phase0_v3", t, HEAD) for t in TASKS}
print("Loaded:", {t: len(v) for t, v in hmaps.items()})

In [ ]:
import matplotlib.gridspec as gridspec

# 2-2-1 layout: rows 0-1 have 2 panels each, row 2 has 1 centred panel.
fig = plt.figure(figsize=(FULL_W, 3 * ROW_H))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.8, wspace=0.4)

axes_flat = [
    fig.add_subplot(gs[0, 0:2]),   # (a) left
    fig.add_subplot(gs[0, 2:4]),   # (b) right
    fig.add_subplot(gs[1, 0:2]),   # (c) left
    fig.add_subplot(gs[1, 2:4]),   # (d) right
    fig.add_subplot(gs[2, 1:3]),   # (e) centred
]

# Per-panel flags: (show_ylabels, show_cbar_label)
#   Left panels  → y-labels ✓, cbar visible but no text label
#   Right panels → y-labels ✗, cbar visible with "AUROC (%)" label
#   Centred lone → y-labels ✓, cbar with label
panel_flags = [
    (True,  False),   # (a) left
    (False, True),    # (b) right
    (True,  False),   # (c) left
    (False, True),    # (d) right
    (True,  True),    # (e) centred
]

for i, (ax, task) in enumerate(zip(axes_flat, TASKS)):
    show_y, show_cbar_lbl = panel_flags[i]
    panels.heatmap_panel(ax, hmaps[task], col=METRIC,
                         show_ylabels=show_y, show_cbar_label=show_cbar_lbl)
    ax.set_title(TASK_LABEL[task], fontsize=8, pad=3)
    add_panel_label(ax, f"({chr(97+i)})")

plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
save_figure(fig, FINAL_OUT, "main_fig3_heatmap")
print("Saved →", FINAL_OUT / "main_fig3_heatmap.pdf")